# Entrenamiento de la tesis en Colab

1. Sube `tesis_colab.zip` a tu Drive, en la carpeta **`MyDrive/tesis`**.
2. *Entorno de ejecución → Cambiar tipo de entorno → GPU* (T4 basta; A100/L4 si tu plan las tiene).
3. *Entorno de ejecución → Ejecutar todas*.

Corre los 7 entrenamientos del protocolo (3 modelos × experimentos A y B, más el control sin máscara), cuantiza cada uno a int8 y lo evalúa. Cada run terminado se guarda en `MyDrive/tesis/runs/`. Si Colab se desconecta, vuelve a *Ejecutar todas*: lo terminado se salta y un run con el modelo ya entrenado retoma desde la cuantización.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/tesis'
!mkdir -p "{DRIVE}/runs"
!nvidia-smi -L

In [ ]:
# Copia al disco local de Colab: leer 58 000 fotos desde Drive en cada época sería lentísimo.
%cd /content
!cp "{DRIVE}/tesis_colab.zip" /content/ && unzip -q -o tesis_colab.zip -d /content/tesis && rm tesis_colab.zip
%cd /content/tesis
import tensorflow as tf
print(tf.__version__, tf.config.list_physical_devices('GPU'))

In [ ]:
import shutil
from pathlib import Path

RUNS = [  # (modelo, experimento, sin máscara)
    ('mobilenetv3', 'A', False), ('efficientnetb0', 'A', False), ('resnet50', 'A', False),
    ('mobilenetv3', 'A', True),  # control del atajo de la superposición
    ('mobilenetv3', 'B', False), ('efficientnetb0', 'B', False), ('resnet50', 'B', False),
]

def sh(cmd):
    print('$', cmd)
    get_ipython().system(cmd)
    if _exit_code:
        raise RuntimeError(f'falló: {cmd}')

for model, exp, nomask in RUNS:
    name = f"{exp}_{model}{'_nomask' if nomask else ''}"
    local, saved = Path('runs') / name, Path(DRIVE) / 'runs' / name
    if (saved / 'metrics_model_int8.json').exists():
        print('ya está:', name)
        continue
    if (saved / 'model.keras').exists():  # entrenado antes de una desconexión: se retoma
        shutil.copytree(saved, local, dirs_exist_ok=True)
    else:
        sh(f'python -m tesis.train --model {model} --exp {exp}' + (' --no-mask' if nomask else ''))
        shutil.copytree(local, saved, dirs_exist_ok=True)
    sh(f'python -m tesis.quantize runs/{name}')
    for f in ('model.keras', 'model_int8.tflite'):  # la diferencia entre ambos es la caída por cuantización
        sh(f'python -m tesis.evaluate runs/{name} {f}')
    shutil.copytree(local, saved, dirs_exist_ok=True)
    print('guardado:', saved)

In [ ]:
# Resumen. Ojo: el conjunto de prueba aún no tiene capturas inválidas reales, así que su recall sale 0 y
# arrastra f1_macro y reject_recall. f1_presentes promedia solo las clases con ejemplos en prueba.
import glob, json
import pandas as pd

rows = []
for f in sorted(glob.glob(f'{DRIVE}/runs/*/metrics_*.json')):
    m = json.load(open(f))
    pc = [v for v in m['per_class'].values() if v['support'] > 0]
    f1 = [2 * v['precision'] * v['recall'] / (v['precision'] + v['recall'] or 1) for v in pc]
    rows.append({'run': f.split('/')[-2], 'formato': f.split('metrics_')[-1][:-5],
                 'f1_macro': m['f1_macro'], 'f1_presentes': sum(f1) / len(f1),
                 'recall_macro': m['recall_macro'], 'reject_recall': m['reject_recall'], 'size_mb': m['size_mb']})
pd.DataFrame(rows).round(3)